In [1]:
!pip install pathway bokeh --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.6/777.6 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.6/244.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.8/985.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
import pandas as pd
import pathway as pw
from bokeh.plotting import figure, curdoc
from bokeh.models import ColumnDataSource
from datetime import datetime

In [3]:
import os
os.makedirs("/content/drive/MyDrive/data_stream", exist_ok=True)

# Data Preprocessing

In [6]:
class ParkingSchema(pw.schema.Schema):
    ID: int
    SystemCodeNumber: str
    Capacity: int
    Latitude: float
    Longitude: float
    Occupancy: int
    VehicleType: str
    TrafficConditionNearby: str
    QueueLength: int
    IsSpecialDay: int
    LastUpdatedDate: str
    LastUpdatedTime: str

In [8]:
table = pw.io.csv.read(
    "/content/drive/MyDrive/data_stream",
    schema=ParkingSchema,
    mode="streaming",
    csv_settings=pw.io.csv.CsvParserSettings(delimiter=","),
    autocommit_duration_ms=100
)

In [9]:
@pw.udf
def combine_datetime(date: str, time: str) -> str:
    return f"{date} {time}"

table_with_timestamp = table.select(
    **table,
    timestamp=combine_datetime(table.LastUpdatedDate, table.LastUpdatedTime)
)

# Model 1

In [10]:
@pw.udf
def occupancy_rate(occupancy: int, capacity: int) -> float:
    return occupancy / capacity if capacity != 0 else 0

@pw.udf
def price_model_1(rate: float) -> str:
    if rate < 0.3:
        return "Low"
    elif rate < 0.7:
        return "Medium"
    else:
        return "High"

model_1_output = table_with_timestamp.select(
    SystemCodeNumber=table_with_timestamp.SystemCodeNumber,
    Occupancy=table_with_timestamp.Occupancy,
    Capacity=table_with_timestamp.Capacity,
    rate=occupancy_rate(table_with_timestamp.Occupancy, table_with_timestamp.Capacity),
    PriceLevel=price_model_1(occupancy_rate(table_with_timestamp.Occupancy, table_with_timestamp.Capacity)),
    Timestamp=table_with_timestamp.timestamp
)

In [11]:
import pathway.io.python as pwio

In [12]:
try:
    df = pd.read_csv("/content/drive/MyDrive/data_stream/dataset.csv")
except FileNotFoundError:
    print("Error:/content/drive/MyDrive/data_stream/dataset.csv")
    df = None

if df is not None:
    # Combine date and time
    df['Timestamp'] = df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime']

    # Calculate occupancy rate
    df['rate'] = df.apply(lambda row: row['Occupancy'] / row['Capacity'] if row['Capacity'] != 0 else 0, axis=1)

    # Determine PriceLevel based on occupancy rate
    def price_model_1_pandas(rate):
        if rate < 0.3:
            return "Low"
        elif rate < 0.7:
            return "Medium"
        else:
            return "High"

    df['PriceLevel'] = df['rate'].apply(price_model_1_pandas)

    # Select the columns for the output
    output_df = df[['SystemCodeNumber', 'Occupancy', 'Capacity', 'rate', 'PriceLevel', 'Timestamp']]

    # Save the output DataFrame to a CSV file
    output_csv_path = "/content/drive/MyDrive/data_stream/output.csv"
    output_df.to_csv(output_csv_path, index=False)

    print(f"Processed data saved to {output_csv_path}")

    # Display the head of the output
    display(output_df.head())

Processed data saved to /content/drive/MyDrive/data_stream/output.csv


,SystemCodeNumber,Occupancy,Capacity,rate,PriceLevel,Timestamp
0,BHMBCCMKT01,61,577,0.105719,Low,04-10-2016 07:59:00
1,BHMBCCMKT01,64,577,0.110919,Low,04-10-2016 08:25:00
2,BHMBCCMKT01,80,577,0.138648,Low,04-10-2016 08:59:00
3,BHMBCCMKT01,107,577,0.185442,Low,04-10-2016 09:32:00
4,BHMBCCMKT01,150,577,0.259965,Low,04-10-2016 09:59:00


In [13]:
output_df.shape

(18368, 6)

In [15]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
import pandas as pd

# Bokeh output in notebook
output_notebook()

# Load the processed data
df = pd.read_csv("/content/drive/MyDrive/data_stream/output.csv")

# Converting Timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
df.dropna(subset=['Timestamp'], inplace=True)

df.sort_values(by='Timestamp', inplace=True)

#plot
p = figure(x_axis_type="datetime", title="Occupancy Over Time by Price Level",
           width=900, height=450, tools="pan,wheel_zoom,box_zoom,reset,hover,save")

price_levels = ['Low', 'Medium', 'High']
colors = Category10[3]

# Plot separate lines for each price level
for i, level in enumerate(price_levels):
    level_df = df[df['PriceLevel'] == level]
    source = ColumnDataSource(level_df)

    p.line(x='Timestamp', y='Occupancy', source=source, legend_label=level,
           line_width=2, color=colors[i], alpha=0.8)

    p.circle(x='Timestamp', y='Occupancy', source=source, size=5, color=colors[i], alpha=0.6)

hover = HoverTool(
    tooltips=[
        ("SystemCode", "@SystemCodeNumber"),
        ("Occupancy", "@Occupancy"),
        ("Capacity", "@Capacity"),
        ("Rate", "@rate{0.00}"),
        ("Price Level", "@PriceLevel"),
        ("Time", "@Timestamp{%F %T}")
    ],
    formatters={"@Timestamp": "datetime"},
    mode='mouse'
)
p.add_tools(hover)

p.xaxis.axis_label = "Timestamp"
p.yaxis.axis_label = "Occupancy"
p.legend.title = "Price Level"
p.legend.location = "top_left"

show(p)


# Model 2

In [16]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/data_stream/dataset.csv")

# Combine date and time into timestamp
df['Timestamp'] = df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime']

# Calculate occupancy rate
df['rate'] = df.apply(lambda row: row['Occupancy'] / row['Capacity'] if row['Capacity'] != 0 else 0, axis=1)

# Define Model 2
def price_model_2(row):
    rate = row['rate']
    queue = row['QueueLength']
    special = row['IsSpecialDay']

    if (rate > 0.7 or queue > 5) and special == 1:
        return "High"
    elif 0.3 < rate <= 0.7:
        return "Medium"
    else:
        return "Low"

#Model 2
df['PriceLevel_Model2'] = df.apply(price_model_2, axis=1)

# Select final output
output_df2 = df[['SystemCodeNumber', 'Occupancy', 'Capacity', 'QueueLength', 'IsSpecialDay', 'rate', 'PriceLevel_Model2', 'Timestamp']]

# Save output to CSV
output_path = "/content/drive/MyDrive/data_stream/output_model2.csv"
output_df2.to_csv(output_path, index=False)

print(f"Model 2 output saved to {output_path}")


Model 2 output saved to /content/drive/MyDrive/data_stream/output_model2.csv


In [19]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
import pandas as pd

output_notebook()

df2 = pd.read_csv("/content/drive/MyDrive/data_stream/output_model2.csv")
df2['Timestamp'] = pd.to_datetime(df2['Timestamp'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
df2.dropna(subset=['Timestamp'], inplace=True)
df2.sort_values(by='Timestamp', inplace=True)

price_levels = ['Low', 'Medium', 'High']
colors = Category10[3]

p = figure(x_axis_type="datetime", title="Model 2: Occupancy Over Time by Price Level",
           width=900, height=450, tools="pan,wheel_zoom,box_zoom,reset,hover,save")

for i, level in enumerate(price_levels):
    level_df = df2[df2['PriceLevel_Model2'] == level]
    source = ColumnDataSource(level_df)
    p.line(x='Timestamp', y='Occupancy', source=source, line_width=2, color=colors[i],
           legend_label=level, alpha=0.8)
    p.circle(x='Timestamp', y='Occupancy', source=source, size=5, color=colors[i], alpha=0.6)

hover = HoverTool(tooltips=[
    ("SystemCode", "@SystemCodeNumber"),
    ("Occupancy", "@Occupancy"),
    ("Capacity", "@Capacity"),
    ("QueueLength", "@QueueLength"),
    ("Special Day", "@IsSpecialDay"),
    ("Rate", "@rate{0.00}"),
    ("Price Level", "@PriceLevel_Model2"),
    ("Time", "@Timestamp{%F %T}")
], formatters={"@Timestamp": "datetime"}, mode='mouse')
p.add_tools(hover)

p.xaxis.axis_label = "Timestamp"
p.yaxis.axis_label = "Occupancy"
p.legend.title = "Price Level"
p.legend.location = "top_left"

show(p)


# Model 3

In [20]:
import pandas as pd
from datetime import datetime

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/data_stream/dataset.csv")

# Combine date and time into timestamp and convert to datetime with correct format and error handling
df['Timestamp'] = df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime']
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d-%m-%Y %H:%M:%S', errors='coerce')

# Drop rows
df.dropna(subset=['Timestamp'], inplace=True)

# Calculate occupancy rate
df['rate'] = df.apply(lambda row: row['Occupancy'] / row['Capacity'] if row['Capacity'] != 0 else 0, axis=1)

# Extract hour from Timestamp
df['Hour'] = df['Timestamp'].dt.hour

# Clean up TrafficConditionNearby column
df['TrafficConditionNearby'] = df['TrafficConditionNearby'].str.strip().str.lower()

# Define Model 3
def price_model_3(row):
    rate = row['rate']
    queue = row['QueueLength']
    traffic = row['TrafficConditionNearby']
    hour = row['Hour']

    if (rate > 0.75 and queue > 5) or (traffic == "heavy"):
        return "High"
    elif 0.3 < rate <= 0.75 or (17 <= hour <= 20):  # 5 PM to 8 PM
        return "Medium"
    else:
        return "Low"

# Apply model 3
df['PriceLevel_Model3'] = df.apply(price_model_3, axis=1)

# Select final output
output_df3 = df[['SystemCodeNumber', 'Occupancy', 'Capacity', 'QueueLength', 'TrafficConditionNearby',
                 'rate', 'Hour', 'PriceLevel_Model3', 'Timestamp']]

# Save output to CSV
output_path = "/content/drive/MyDrive/data_stream/output_model3.csv"
output_df3.to_csv(output_path, index=False)

print(f"Model 3 output saved to {output_path}")

Model 3 output saved to /content/drive/MyDrive/data_stream/output_model3.csv


In [23]:
output_df2.shape

(18368, 8)

In [24]:
output_df3.shape

(18368, 9)